In [18]:
import spacy
import os
import requests
import re
import io
from Bio import Entrez

In [25]:
Entrez.email = "nicolekenneth214@gmail.com"
Entrez.api_key = os.getenv("NCBI_API_KEY")

def get_variant_articles_ncbi_only(hgvs_query):
    # Search PubMed for HGVS string
    search_term = f'"{hgvs_query}"'
    
    handle = Entrez.esearch(db="pubmed", term=search_term, retmax=5)
    record = Entrez.read(handle)
    handle.close()


    ids = record["IdList"]

    if not ids:
        print(f"No exact matches found in PubMed for {hgvs_query}")
        return None

    # Fetch XML for the articles
    fetch_handle = Entrez.efetch(db="pubmed", id=",".join(ids), retmode="xml")
    papers_xml = fetch_handle.read()
    fetch_handle.close()
    
    return papers_xml


# Execution
query = "BRCA1"
raw_xml = get_variant_articles_ncbi_only(query)

if raw_xml:
    with open("data/papers.xml", "wb") as f:
        f.write(raw_xml)
    print("Saved XML")


"""
#1. Download papers from PubMed Central

from Bio import Entrez
#Entrez.email = "jawahir_noor@hotmail.com"

handle = Entrez.esearch(db="PMC", term="genomic variants AND mutation AND human", retmax=8)
record = Entrez.read(handle)

ids = record["IdList"]

#using PubMed Central (PMC) to get full articles
fetch = Entrez.efetch(db="PMC", id=ids, rettype="abstract", retmode="text")
papers = fetch.read()

#print(papers)
# SAVE TO FILE
with open("data/papers.txt", "w", encoding="utf-8") as f:
    f.write(papers)

print("Saved to papers.txt ✅")
"""

Saved XML


'\n#1. Download papers from PubMed Central\n\nfrom Bio import Entrez\n#Entrez.email = "jawahir_noor@hotmail.com"\n\nhandle = Entrez.esearch(db="PMC", term="genomic variants AND mutation AND human", retmax=8)\nrecord = Entrez.read(handle)\n\nids = record["IdList"]\n\n#using PubMed Central (PMC) to get full articles\nfetch = Entrez.efetch(db="PMC", id=ids, rettype="abstract", retmode="text")\npapers = fetch.read()\n\n#print(papers)\n# SAVE TO FILE\nwith open("data/papers.txt", "w", encoding="utf-8") as f:\n    f.write(papers)\n\nprint("Saved to papers.txt ✅")\n'

In [ ]:
# Setup
Entrez.email = "nicolekenneth214@gmail.com"
Entrez.api_key = os.getenv("NCBI_API_KEY")
litvar_api = "https://www.ncbi.nlm.nih.gov/research/litvar2-api"

def get_articles(hgvs_string):
    # Regex to anchor the HGVS or variant string
    pattern = r"(?:[cp]\.\d+[A-Z>_delinsdup]+|rs\d+|[A-Z][0-9]+\s+[cp]\.\d+[A-Z>_delinsdup]+)"
    match_found = re.search(pattern, hgvs_string, re.IGNORECASE)

    if not match_found:
        print("Not valid variant notation")
        return None

    clean_query = match_found.group(0)
    print(f"Regex anchored to: {clean_query}")

    # Get IDs from Litvar
    search_url = f"{litvar_api}/variant/autocomplete/"
    search_res = requests.get(search_url, params={"query": clean_query})

    if search_res.status_code != 200:
        print(f"LitVar Error: Server returned {search_res.status_code}")
        return None

    try:
        search_data = search_res.json()
        variant_id = search_data.get('id')
    except Exception:
        print("Error: Could not parse LitVar response.")
        return None
    
    if not variant_id:
        print(f"LitVar: Could not map '{clean_query}' to an internal ID.")
        return None
    
    # Get PMIDs
    pub_url = f"{litvar_api}/variant/get/{variant_id}/publications"
    pub_res = requests.get(pub_url)
    pmids = pub_res.json().get('pmids', [])

    if not pmids:
        print(f"LitVar: No papers found linked to ID {variant_id}")
        return None
    
    # Fetch XML from PubMed
    pmid_string = ",".join(pmids[:5])
    print(f"Fetching records for: {pmid_string}")
    
    handle = Entrez.efetch(db="pubmed", id=pmid_string, retmode="xml")
    raw_xml = handle.read()
    handle.close()
    
    return raw_xml

# Execution
user_input = "NM_007294.4:c.5266dupC"
xml_data = get_articles(user_input)

if xml_data:
    os.makedirs("data", exist_ok=True)
    
    # Parse the raw bytes into a dictionary
    records = Entrez.read(io.BytesIO(xml_data))
    articles = records.get('PubmedArticle', [])

    with open("data/papers.xml", "w", encoding="utf-8") as f:
        for i, article in enumerate(articles):
            f.write(f"--- ARTICLE {i+1} ---\n")
            f.write(str(article))
            f.write("\n\n" + "="*60 + "\n\n")

    print("PMIDs fetched and saved to papers.xml")



Regex anchored to: c.5266dupC
Error: Could not parse LitVar response.


'\ndef get_variant_articles_ncbi_only(hgvs_query):\n    # Search PubMed for HGVS string\n    search_term = f\'"{hgvs_query}"\'\n\n    handle = Entrez.esearch(db="pubmed", term=search_term, retmax=5)\n    record = Entrez.read(handle)\n    handle.close()\n\n    # This is the \'ids\' list you wanted\n    ids = record["IdList"]\n\n    if not ids:\n        print(f"No exact matches found in PubMed for {hgvs_query}")\n        return None\n\n    # Now fetch the actual XML for those IDs\n    fetch_handle = Entrez.efetch(db="pubmed", id=",".join(ids), retmode="xml")\n    papers_xml = fetch_handle.read()\n    fetch_handle.close()\n\n    return papers_xml\n\n\n# Execution\nquery = "BRCA1"\nraw_xml = get_variant_articles_ncbi_only(query)\n\nif raw_xml:\n    with open("data/papers.xml", "wb") as f:\n        f.write(raw_xml)\n    print("Saved XML using NCBI Search ✅")\n'

In [34]:
#2.Process the text with scispaCy
import spacy
import pandas as pd

nlp = spacy.load("en_core_sci_sm")

papers = raw_xml.decode("utf-8")

results = []

# Step 1: split abstracts
chunks = papers.split("\n\n")

MAX_LEN = 5000

for chunk in chunks:
    chunk = chunk.strip()

    if len(chunk) == 0:
        continue

    sub_chunks = [chunk[i:i+MAX_LEN] for i in range(0, len(chunk), MAX_LEN)]

    for sub in sub_chunks:
        doc = nlp(sub)

        for ent in doc.ents:
            results.append({
                "text": ent.text,
                "label": ent.label_
            })

# Step 2: dataframe
df = pd.DataFrame(results)

print(df.head())

# Step 3: save
df.to_csv("scispacy_entities.csv", index=False)

OSError: [E050] Can't find model 'en_core_sci_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

In [ ]:
#3.Extract biomedical entities (BioNLP13CG + scispaCy)
doc_sci = nlp_sci(doc)
doc_bio = nlp_bionlp(doc)

In [ ]:
#4. Extract related entities with scispaCy